In [81]:
import re
from typing import List, Optional, Tuple
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, ConfusionMatrixDisplay, confusion_matrix,
    balanced_accuracy_score,
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

# 🎯 XGBOOST IMPORT
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not available. Install with: pip install xgboost")


# ============================================================
# ALIGNMENT CHANGES APPLIED
# ============================================================
# 1) STATUS SETS unified with the other pipelines.
# 2) Unbounded rows are kept for CLASSIFICATION labels, but excluded from delta:
#       - y is assigned (LP/SDP)
#       - delta stays NaN
#       - they are not treated as comparable delta rows
#       - sample-weight strength for them defaults to base weight only
# 3) LP-biased binary decision everywhere:
#       choose LP iff delta > -TAU_BINARY
# 4) Sample weights aligned with regression v2:
#       w = (1 - WEIGHT_ALPHA) + WEIGHT_ALPHA * |strength|
#       binary strength    = |delta_raw| for finite-comparable rows, NaN otherwise
#       multiclass strength = normalized margin between best and second-best finite objectives
# ============================================================


# ============================================================
# FEATURE FILE CONFIG
# ============================================================
FEATURE_FILE = "selected_featuresclassdummy.txt"


# ============================================================
# CONFIG
# ============================================================
SETUP = "DI"

TRAIN_FEATURES = ["features_5_10_10a_DI_synth.xlsx"]
TRAIN_OBJECTS  = ["instancesobj_5_10_10a.csv"]

TEST_FEATURES = ["features_10_2a_DI_synth.xlsx"]
TEST_OBJECTS  = ["instancesobj_10_2a.csv"]

USE_HOLDOUT_FROM_TRAIN = (len(TEST_FEATURES) == 0)

KEY_COL = "id_key"


# ============================================================
# CLASSIFICATION MODE
# ============================================================
MODE = "binary"   # "binary" or "multiclass"

RELAXATION_PAIR = ("lp", "sdp_prime")
TAU_BINARY = 0.01
EPSILON_GAP = 1e-3

RELAXATIONS = ["lp", "sdp", "sdp_prime", "sdp_rlt"]


# ============================================================
# SAMPLE WEIGHTS  [ALIGNED]
# ============================================================
USE_SAMPLE_WEIGHTS = True
WEIGHT_ALPHA = 0.30   # w = (1-alpha) + alpha*|strength|


# ============================================================
# STATUS SETS  [UNIFIED]
# ============================================================
OPTIMAL_STATUSES = frozenset({
    "optimal",
    "optimal_inaccurate",
})

UNBOUNDED_STATUSES = frozenset({
    "unbounded",
    "unbounded_inaccurate",
    "unbounded_uncertified",
})

EXCLUDE_STATUSES = frozenset({
    "infeasible", "infeasible_inaccurate", "infeasible_or_unbounded",
    "primal_infeasible", "dual_infeasible_cert", "unknown",
    "numeric_error", "time_limit", "iteration_limit", "node_limit",
    "interrupted", "memory_limit",
})

KEEP_UNKNOWN = "drop"

OUTPUT_ROOT = Path("clf_outputs_txt_features_synthmissings")
OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)
RANDOM_STATE = 42


# ============================================================
# HELPERS
# ============================================================
def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def infer_tag(path: str) -> str:
    stem = Path(path).stem
    m = re.search(r'(\d+_\d+)(?:$|_)', stem)
    if m:
        return m.group(1)
    parts = stem.split("_", 1)
    return parts[1] if len(parts) > 1 else stem


def run_tag_from_lists(fpaths: List[str]) -> str:
    tags = sorted({infer_tag(p) for p in fpaths})
    return "+".join(tags) if tags else "unk"


def read_any(path: str) -> pd.DataFrame:
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(p)
    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    raise ValueError(f"Unsupported file type: {p.suffix}")


def _norm_status(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x).strip().lower()


# ============================================================
# DETERMINISTIC SPLIT  (same hash logic as GNN and regression)
# ============================================================
def make_deterministic_split(keys: pd.Series, test_frac: float = 0.2):
    """
    Hash-based 80/20 split using pandas key hashing.
    Bucket = hash(str(key)) % 100.
    train : bucket < 80
    test  : bucket >= 80
    """
    h = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b = (h % 100).astype(np.int64)
    cut = int((1.0 - test_frac) * 100)
    tr_idx = np.where(b < cut)[0]
    te_idx = np.where(b >= cut)[0]
    return tr_idx, te_idx


# ============================================================
# FEATURE FILE
# ============================================================
def load_features_from_txt(feature_file: str) -> Optional[List[str]]:
    if feature_file is None:
        print(f"[{_ts()}] [features] No feature file - using ALL features", flush=True)
        return None
    fpath = Path(feature_file)
    if not fpath.exists():
        print(f"[{_ts()}] [features] ⚠️  File not found: {feature_file} - using ALL", flush=True)
        return None
    try:
        with open(fpath, "r") as f:
            features = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]
        print(f"[{_ts()}] [features] ✅ Loaded {len(features)} features from {feature_file}", flush=True)
        for i, feat in enumerate(features[:10], 1):
            print(f"[{_ts()}]   {i:2d}. {feat}", flush=True)
        if len(features) > 10:
            print(f"[{_ts()}]   ... and {len(features) - 10} more", flush=True)
        return features
    except Exception as e:
        print(f"[{_ts()}] [features] ⚠️  Error: {e} - using ALL", flush=True)
        return None


def filter_to_selected_features(X: pd.DataFrame, selected_features: Optional[List[str]]) -> pd.DataFrame:
    if selected_features is None:
        print(f"[{_ts()}] [features] Using ALL {X.shape[1]} features", flush=True)
        return X
    available = [f for f in selected_features if f in X.columns]
    missing   = [f for f in selected_features if f not in X.columns]
    if missing:
        print(f"[{_ts()}] [features] ⚠️  {len(missing)} features not found", flush=True)
        for f in missing[:5]:
            print(f"[{_ts()}]   - {f}", flush=True)
        if len(missing) > 5:
            print(f"[{_ts()}]   ... and {len(missing)-5} more", flush=True)
    if len(available) == 0:
        print(f"[{_ts()}] [features] ❌ NO selected features found - using ALL", flush=True)
        return X
    print(f"[{_ts()}] [features] Using {len(available)}/{len(selected_features)} features", flush=True)
    return X[available]


# ============================================================
# MERGE
# ============================================================
def merge_features_objectives(Xf, Yf, prefer="id_global", required_obj_cols=None):
    Xf = Xf.copy(); Yf = Yf.copy()

    if ("id_global" not in Xf.columns) and ("id" in Xf.columns) and ("id_global" in Yf.columns):
        if Xf["id"].astype(str).str.contains("__").any():
            Xf["id_global"] = Xf["id"].astype(str)

    if ("id_global" not in Yf.columns) and ("id" in Yf.columns) and ("id_global" in Xf.columns):
        if Yf["id"].astype(str).str.contains("__").any():
            Yf["id_global"] = Yf["id"].astype(str)

    key = None
    for k in [prefer, "id_global", "id"]:
        if (k in Xf.columns) and (k in Yf.columns):
            key = k
            break

    if key is None:
        raise ValueError(
            f"No common merge key.\n"
            f"X cols: {list(Xf.columns)[:12]}\n"
            f"Y cols: {list(Yf.columns)[:12]}"
        )

    if key == "id":
        Xf[key] = pd.to_numeric(Xf[key], errors="coerce").astype("Int64")
        Yf[key] = pd.to_numeric(Yf[key], errors="coerce").astype("Int64")
    else:
        Xf[key] = Xf[key].astype(str).str.strip()
        Yf[key] = Yf[key].astype(str).str.strip()

    if required_obj_cols:
        miss = [c for c in required_obj_cols if c not in Yf.columns]
        if miss:
            raise ValueError(f"Objectives missing: {miss}\nAvailable: {list(Yf.columns)[:25]}")

    cols_to_merge = [key] + (required_obj_cols or [])
    for obj_col in (required_obj_cols or []):
        if obj_col.endswith("_obj"):
            status_col = obj_col.replace("_obj", "_status")
            if status_col in Yf.columns:
                cols_to_merge.append(status_col)

    df = pd.merge(Xf, Yf[cols_to_merge], on=key, how="inner")
    df["id_key"] = df[key].astype(str).str.strip()
    return df, key


def load_merge_one(features_path, objectives_path, prefer_key="id_global", required_obj_cols=None):
    Xf = read_any(features_path)
    Yf = read_any(objectives_path)
    df, used_key = merge_features_objectives(Xf, Yf, prefer=prefer_key, required_obj_cols=required_obj_cols)
    df["_source_feat"] = Path(features_path).name
    df["_source_obj"]  = Path(objectives_path).name
    df["_merge_key"]   = used_key
    print(f"[{_ts()}] [merge] {Path(features_path).name} + {Path(objectives_path).name} -> rows={len(df)}", flush=True)
    if len(df) == 0:
        print(f"[{_ts()}] [merge][WARN] 0 rows!", flush=True)
    return df


def concat_pairs(feature_paths, objective_paths, prefer_key="id_global", required_obj_cols=None):
    if len(feature_paths) != len(objective_paths):
        raise ValueError("Feature and objective file counts must match.")
    frames = [load_merge_one(fx, oy, prefer_key=prefer_key, required_obj_cols=required_obj_cols)
              for fx, oy in zip(feature_paths, objective_paths)]
    df = pd.concat(frames, ignore_index=True)
    if "id_key" not in df.columns:
        raise ValueError("Internal error: 'id_key' not created.")
    return df


# ============================================================
# COLUMN DROPS / FILTERS
# ============================================================
def drop_unused_relaxation_cols(df, used_relaxations):
    df = df.copy()
    all_relaxations = ["lp", "sdp", "sdpp", "sdp_rlt", "sdp_prime"]
    unused = [r for r in all_relaxations if r not in used_relaxations]
    to_drop = sorted(set(c for r in unused for c in df.columns if c.startswith(f"{r}_")))
    if to_drop:
        df = df.drop(columns=to_drop, errors="ignore")
        print(f"[{_ts()}] [filter] Dropped {len(to_drop)} cols for unused relaxations: {unused}", flush=True)
    return df


def drop_metadata_cols(df):
    df = df.copy()
    to_drop = sorted(set(
        [c for c in df.columns if str(c).endswith(("_runtime", "_t", "_status", "_relaxation"))]
        + [c for c in df.columns if str(c).startswith(("bt_", "bigM_", "san_"))]
        + [c for c in df.columns if str(c) in ("n_inf_after_bt", "Iteration")]
    ))
    if to_drop:
        df = df.drop(columns=to_drop, errors="ignore")
    return df


def filter_by_status(df, relaxations, exclude_statuses):
    """
    Only EXCLUDE_STATUSES are filtered out here.
    Unbounded rows are intentionally NOT dropped:
      - they stay available for classification labels
      - they are excluded from delta/comparable strength by keeping delta = NaN
    """
    df = df.copy()
    n_before = len(df)

    bad = pd.Series(False, index=df.index)

    for r in relaxations:
        sc = f"{r}_status"
        oc = f"{r}_obj"

        if sc in df.columns:
            bad |= df[sc].map(_norm_status).isin(exclude_statuses)

        if oc in df.columns:
            status = df[sc].map(_norm_status) if sc in df.columns else pd.Series("", index=df.index)
            bad |= (status == "") & df[oc].isna()

    df = df[~bad].copy()
    removed = n_before - len(df)
    if removed:
        print(f"[{_ts()}] [filter] Removed {removed} rows with excluded statuses ({len(df)} remain)", flush=True)
    return df


# ============================================================
# LABELING / STRENGTH
# ============================================================
def add_label_binary(df, relax_a, relax_b, tau=0.01, epsilon_gap=1e-3, keep_unknown="drop"):
    """
    LP-biased binary labeling:
        choose relax_a (LP) iff delta > -tau

    Unbounded handling:
        relax_b unbounded + relax_a finite  -> y=1, delta stays NaN
        relax_a unbounded + relax_b finite  -> y=0, delta stays NaN
        both finite-comparable              -> delta computed, y from delta > -tau
        otherwise                           -> unlabeled -> dropped if keep_unknown='drop'
    """
    out = df.copy()

    col_a = f"{relax_a}_obj"
    col_b = f"{relax_b}_obj"
    st_a  = f"{relax_a}_status"
    st_b  = f"{relax_b}_status"

    out[col_a] = pd.to_numeric(out.get(col_a), errors="coerce")
    out[col_b] = pd.to_numeric(out.get(col_b), errors="coerce")

    a_val = out[col_a].astype(float)
    b_val = out[col_b].astype(float)

    a_stat = out[st_a].map(_norm_status) if st_a in out.columns else pd.Series([""] * len(out), index=out.index)
    b_stat = out[st_b].map(_norm_status) if st_b in out.columns else pd.Series([""] * len(out), index=out.index)

    a_unb = a_stat.isin(UNBOUNDED_STATUSES) | np.isneginf(a_val)
    b_unb = b_stat.isin(UNBOUNDED_STATUSES) | np.isneginf(b_val)

    a_fin = np.isfinite(a_val) & ~a_unb
    b_fin = np.isfinite(b_val) & ~b_unb

    out["y"] = np.nan
    out["delta"] = np.nan
    out["delta_raw_for_weight"] = np.nan
    out["target_case"] = ""

    # One-sided unbounded -> keep classification label, but exclude from delta strength
    out.loc[b_unb & a_fin, "y"] = 1
    out.loc[b_unb & a_fin, "target_case"] = f"{relax_b}_unbounded"

    out.loc[a_unb & b_fin, "y"] = 0
    out.loc[a_unb & b_fin, "target_case"] = f"{relax_a}_unbounded"

    # In add_label_binary(), after assigning y for unbounded rows:
    out.loc[b_unb & a_fin, "delta_raw_for_weight"] = 1.0   # LP wins clearly
    out.loc[a_unb & b_fin, "delta_raw_for_weight"] = -1.0  # mapped to |1| = 1.0 for weight

    # Finite comparable -> compute aligned fixed-additive delta
    finite = a_fin & b_fin
    if finite.any():
        den = (a_val[finite].abs() + b_val[finite].abs()) + float(epsilon_gap)
        delta = (a_val[finite] - b_val[finite]) / den
        out.loc[finite, "delta"] = delta
        out.loc[finite, "delta_raw_for_weight"] = delta
        out.loc[finite, "y"] = (delta > -tau).astype(int)
        out.loc[finite, "target_case"] = "finite"
        print(f"[{_ts()}] [label] LP-biased rule: choose {relax_a} iff delta > -{tau}", flush=True)
        print(f"[{_ts()}] [label] EPSILON_GAP={epsilon_gap}", flush=True)

    decided = out["y"].notna()
    n_undecided = int((~decided).sum())

    if keep_unknown.lower() == "drop":
        out = out.loc[decided].copy()
        if n_undecided:
            print(f"[{_ts()}] [label] Dropped {n_undecided} unlabeled instances", flush=True)
    else:
        out.loc[~decided, "y"] = 1
        out.loc[~decided, "target_case"] = "unknown_assigned_lp"

    out["y"] = out["y"].astype(int)

    vc = out["y"].value_counts().to_dict()
    print(f"[{_ts()}] [label] y=1({relax_a})={vc.get(1,0)}, y=0({relax_b})={vc.get(0,0)}", flush=True)
    print(f"[{_ts()}] [label] case counts: {out['target_case'].value_counts().to_dict()}", flush=True)

    return out


def add_label_multiclass(df, relaxations, keep_unknown="drop"):
    out = df.copy()

    obj_cols = [f"{r}_obj" for r in relaxations]
    for col in obj_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    obj_matrix = np.full((len(out), len(relaxations)), np.nan)
    for i, col in enumerate(obj_cols):
        if col in out.columns:
            obj_matrix[:, i] = out[col].values

    # Treat unbounded as unusable for multiclass argmax
    for i, r in enumerate(relaxations):
        sc = f"{r}_status"
        if sc in out.columns:
            status = out[sc].map(_norm_status)
            obj_matrix[status.isin(UNBOUNDED_STATUSES).values | np.isneginf(obj_matrix[:, i]), i] = -np.inf
        else:
            obj_matrix[np.isneginf(obj_matrix[:, i]), i] = -np.inf

    with np.errstate(invalid='ignore'):
        y = np.nanargmax(obj_matrix, axis=1)

    has_finite = np.any(np.isfinite(obj_matrix), axis=1)
    y[~has_finite] = -1

    out["y"] = y

    undecided = (out["y"] == -1)
    n_undecided = int(undecided.sum())

    if keep_unknown.lower() == "drop":
        out = out.loc[~undecided].copy()
        if n_undecided:
            print(f"[{_ts()}] [label] Dropped {n_undecided} unlabeled", flush=True)
    else:
        out.loc[undecided, "y"] = 0

    out["y"] = out["y"].astype(int)

    vc = out["y"].value_counts().to_dict()
    label_str = ", ".join([f"y={i}({relaxations[i]})={vc.get(i,0)}" for i in range(len(relaxations))])
    print(f"[{_ts()}] [label] {label_str}", flush=True)

    return out


def _compute_multiclass_strength(df: pd.DataFrame, relaxations: List[str], epsilon_gap: float) -> np.ndarray:
    """
    Logical aligned multiclass strength for sample weights:
      margin = (best_finite_obj - second_best_finite_obj) /
               (|best| + |second| + epsilon_gap)

    If fewer than 2 finite usable objectives exist -> NaN -> base weight only.
    """
    obj_mat = []
    for r in relaxations:
        oc = f"{r}_obj"
        sc = f"{r}_status"
        vals = pd.to_numeric(df.get(oc), errors="coerce").astype(float).to_numpy()
        if sc in df.columns:
            st = df[sc].map(_norm_status).to_numpy()
            unusable = np.isin(st, list(EXCLUDE_STATUSES | UNBOUNDED_STATUSES)) | np.isneginf(vals)
            vals = vals.copy()
            vals[unusable] = -np.inf
        else:
            vals = vals.copy()
            vals[np.isneginf(vals)] = -np.inf
        obj_mat.append(vals)

    M = np.column_stack(obj_mat)
    strength = np.full(M.shape[0], np.nan)

    for i in range(M.shape[0]):
        row = M[i]
        finite_vals = row[np.isfinite(row)]
        if finite_vals.size >= 2:
            best = np.max(finite_vals)
            second = np.partition(finite_vals, -2)[-2]
            strength[i] = (best - second) / (abs(best) + abs(second) + float(epsilon_gap))

    return strength


def _make_sample_weights_from_strength(strength: np.ndarray, alpha: float) -> np.ndarray:
    a = min(max(float(alpha), 0.0), 1.0)
    s = np.abs(np.asarray(strength, dtype=float))
    s = np.where(np.isfinite(s), s, 0.0)
    return (1.0 - a) + a * s


# ============================================================
# PREPROCESSING
# ============================================================
def build_preprocessor(X):
    sparsity_cols = [c for c in X.columns if str(c).startswith("Sparsity")]
    num_cols = [
        c for c in X.columns
        if c not in sparsity_cols and (pd.api.types.is_numeric_dtype(X[c]) or pd.api.types.is_bool_dtype(X[c]))
    ]
    try:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

    pre = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe),
        ]), sparsity_cols),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
        ]), num_cols),
    ], remainder="drop")

    return pre, sparsity_cols, num_cols


def align_test_to_train_cols(X_train, X_test):
    X_test = X_test.copy()
    for c in [c for c in X_train.columns if c not in X_test.columns]:
        X_test[c] = np.nan
    return X_test[X_train.columns]


# ============================================================
# TRAIN / EVAL
# ============================================================
def train_eval_classification(
    train_df, test_df, key, run_dir, run_tag,
    mode, relaxations, tau_binary=0.01, epsilon_gap=1e-3,
    selected_features=None,
):
    print(f"[{_ts()}] ═══════════════════════════════════════", flush=True)
    if mode == "binary":
        print(f"[{_ts()}] BINARY: {relaxations[0]} (y=1) vs {relaxations[1]} (y=0)", flush=True)
        print(f"[{_ts()}] TAU={tau_binary}  EPSILON_GAP={epsilon_gap}", flush=True)
        print(f"[{_ts()}] DECISION RULE: choose {relaxations[0]} iff delta > -{tau_binary}", flush=True)
    else:
        print(f"[{_ts()}] MULTICLASS: {len(relaxations)} classes", flush=True)

    n_feat_str = f"{len(selected_features)} selected" if selected_features else "ALL"
    print(f"[{_ts()}] FEATURES: {n_feat_str}", flush=True)
    print(f"[{_ts()}] SAMPLE_WEIGHTS={USE_SAMPLE_WEIGHTS}  WEIGHT_ALPHA={WEIGHT_ALPHA}", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════", flush=True)

    train_df = filter_by_status(train_df, relaxations, EXCLUDE_STATUSES)
    if test_df is not None:
        test_df = filter_by_status(test_df, relaxations, EXCLUDE_STATUSES)

    if mode == "binary":
        train_df = add_label_binary(
            train_df, relaxations[0], relaxations[1],
            tau=tau_binary, epsilon_gap=epsilon_gap, keep_unknown=KEEP_UNKNOWN
        )
        if test_df is not None:
            test_df = add_label_binary(
                test_df, relaxations[0], relaxations[1],
                tau=tau_binary, epsilon_gap=epsilon_gap, keep_unknown=KEEP_UNKNOWN
            )
        train_strength_all = train_df["delta_raw_for_weight"].astype(float).to_numpy()
    else:
        train_df = add_label_multiclass(train_df, relaxations=relaxations, keep_unknown=KEEP_UNKNOWN)
        if test_df is not None:
            test_df = add_label_multiclass(test_df, relaxations=relaxations, keep_unknown=KEEP_UNKNOWN)
        train_strength_all = _compute_multiclass_strength(train_df, relaxations, epsilon_gap)

    train_df = drop_unused_relaxation_cols(train_df, relaxations)
    if test_df is not None:
        test_df = drop_unused_relaxation_cols(test_df, relaxations)

    train_df = drop_metadata_cols(train_df)
    if test_df is not None:
        test_df = drop_metadata_cols(test_df)

    if len(train_df) == 0:
        raise ValueError("train_df became 0 rows after filtering/labeling")

    ID_LIKE = ["id", "id_global", "instance_id", "instanceid", "instance_name"]
    obj_cols = [f"{r}_obj" for r in relaxations]
    extra_drop = ["delta", "y", "_source_feat", "_source_obj", "_merge_key", "delta_raw_for_weight", "target_case"]
    drop_cols = sorted(set(
        [c for c in ([key] + obj_cols + extra_drop) if c in train_df.columns]
        + [c for c in ID_LIKE if c in train_df.columns]
    ))

    Xtr = train_df.drop(columns=drop_cols, errors="ignore")
    ytr = train_df["y"].astype(int)

    Xtr = filter_to_selected_features(Xtr, selected_features)
    Xtr = Xtr.replace([np.inf, -np.inf], np.nan)

    ids_tr = train_df.loc[Xtr.index, key].astype(str)

    # Reset so iloc-based deterministic split stays perfectly aligned
    Xtr = Xtr.reset_index(drop=True)
    ytr = ytr.reset_index(drop=True)
    ids_tr = ids_tr.reset_index(drop=True)
    train_strength_all = np.asarray(train_strength_all, dtype=float)
    if len(train_strength_all) != len(Xtr):
        raise ValueError("Internal alignment error: strength vector length mismatch.")
    train_strength_all = train_strength_all[:len(Xtr)]

    print(f"[{_ts()}] [features] Train shape: {Xtr.shape}", flush=True)

    n_classes = len(np.unique(ytr))
    if n_classes < 2:
        raise ValueError("Train labels are single-class")

    pre, _, _ = build_preprocessor(Xtr)

    def _make_models():
        models = {
            "hgbc": HistGradientBoostingClassifier(
                learning_rate=0.08,
                max_depth=None,
                max_iter=400,
                early_stopping=True,
                validation_fraction=0.1,
                l2_regularization=1e-3,
                random_state=RANDOM_STATE
            ),
            "gbc": GradientBoostingClassifier(random_state=RANDOM_STATE),
            "rf": RandomForestClassifier(
                n_estimators=400,
                max_features="sqrt",
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE
            ),
        }

        if XGBOOST_AVAILABLE:
            eval_m = "logloss" if mode == "binary" else "mlogloss"
            models["xgb"] = XGBClassifier(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric=eval_m
            )
            print(f"[{_ts()}] [models] ✅ XGBoost enabled", flush=True)
        else:
            print(f"[{_ts()}] [models] ⚠️  XGBoost not available", flush=True)

        return models

    models = _make_models()
    results = {}

    def _eval_and_save(name, pipe, X_te, y_te, ids_te, n_train_used: int):
        y_hat = pipe.predict(X_te)

        row = {
            "model": name,
            "mode": mode,
            "n_features": Xtr.shape[1],
            "n_classes": n_classes,
            "n_train": int(n_train_used),
            "n_test": int(len(y_te)),
            "accuracy": float(accuracy_score(y_te, y_hat)),
            "sample_weights": USE_SAMPLE_WEIGHTS,
            "weight_alpha": WEIGHT_ALPHA,
        }

        if mode == "binary":
            row["f1"]        = float(f1_score(y_te, y_hat, zero_division=0))
            row["f1_macro"]  = float(f1_score(y_te, y_hat, average="macro", zero_division=0))
            row["bal_acc"]   = float(balanced_accuracy_score(y_te, y_hat))
            row["precision"] = float(precision_score(y_te, y_hat, zero_division=0))
            row["recall"]    = float(recall_score(y_te, y_hat, zero_division=0))
            if hasattr(pipe, "predict_proba") and np.unique(y_te).size > 1:
                y_prob = pipe.predict_proba(X_te)[:, 1]
                row["roc_auc"] = float(roc_auc_score(y_te, y_prob))
        else:
            row["f1_macro"]    = float(f1_score(y_te, y_hat, average="macro", zero_division=0))
            row["f1_weighted"] = float(f1_score(y_te, y_hat, average="weighted", zero_division=0))
            row["bal_acc"]     = float(balanced_accuracy_score(y_te, y_hat))
            if hasattr(pipe, "predict_proba") and n_classes > 1:
                try:
                    y_prob = pipe.predict_proba(X_te)
                    row["roc_auc_ovr"] = float(roc_auc_score(y_te, y_prob, multi_class="ovr"))
                except Exception:
                    pass

        metric_str = f"acc={row['accuracy']:.3f} bal_acc={row['bal_acc']:.3f}"
        if mode == "binary":
            metric_str += f" f1_macro={row['f1_macro']:.3f} auc={row.get('roc_auc', 'N/A')}"
        else:
            metric_str += f" f1_macro={row['f1_macro']:.3f}"
        print(f"[{_ts()}] [eval] {name}: {metric_str}", flush=True)

        pred_df = pd.DataFrame({key: ids_te.astype(str), "y_true": y_te, "y_pred": y_hat})
        if hasattr(pipe, "predict_proba"):
            y_prob = pipe.predict_proba(X_te)
            if mode == "binary":
                pred_df[f"y_prob_{relaxations[0]}"] = y_prob[:, 1]
            else:
                for i, r in enumerate(relaxations):
                    pred_df[f"y_prob_{r}"] = y_prob[:, i]
        pred_df.to_csv(run_dir / f"preds_{name}_{run_tag}.csv", index=False)

        labels = list(range(n_classes))
        cm = confusion_matrix(y_te, y_hat, labels=labels)

        if mode == "binary":
            index_labels = [f"true_{relaxations[1]}(0)", f"true_{relaxations[0]}(1)"]
            column_labels = [f"pred_{relaxations[1]}(0)", f"pred_{relaxations[0]}(1)"]
            display_labels = [relaxations[1], relaxations[0]]
        else:
            index_labels = [f"true_{relaxations[i]}({i})" for i in range(n_classes)]
            column_labels = [f"pred_{relaxations[i]}({i})" for i in range(n_classes)]
            display_labels = [relaxations[i] for i in range(n_classes)]

        pd.DataFrame(cm, index=index_labels, columns=column_labels).to_csv(
            run_dir / f"cm_{name}_{run_tag}.csv", index=True
        )

        disp = ConfusionMatrixDisplay(cm, display_labels=display_labels)
        fig, ax = plt.subplots(figsize=(max(8, n_classes * 2), max(6, n_classes * 1.5)))
        disp.plot(ax=ax)
        plt.tight_layout()
        plt.savefig(run_dir / f"cm_{name}_{run_tag}.png", dpi=200)
        plt.close()

        return row

    # ========================================================
    # HOLDOUT
    # ========================================================
    if test_df is None:
        tr_idx, te_idx = make_deterministic_split(ids_tr, test_frac=0.2)
        X_tr = Xtr.iloc[tr_idx]
        X_te = Xtr.iloc[te_idx]
        y_tr = ytr.iloc[tr_idx]
        y_te = ytr.iloc[te_idx]
        ids_te = ids_tr.iloc[te_idx]

        w_tr = None
        if USE_SAMPLE_WEIGHTS:
            w_tr = _make_sample_weights_from_strength(train_strength_all[tr_idx], WEIGHT_ALPHA)

        print(f"[{_ts()}] [split] Holdout (hash-based): train={len(y_tr)}, test={len(y_te)}", flush=True)

        for name, mdl in models.items():
            print(f"[{_ts()}] [train] Training {name}...", flush=True)
            pipe = Pipeline([("pre", pre), ("model", mdl)])
            fit_kwargs = {"model__sample_weight": w_tr} if w_tr is not None else {}
            pipe.fit(X_tr, y_tr, **fit_kwargs)
            results[name] = _eval_and_save(name, pipe, X_te, y_te, ids_te, len(y_tr))

    # ========================================================
    # EXTERNAL TEST
    # ========================================================
    else:
        if len(test_df) == 0:
            raise ValueError("test_df has 0 rows after filtering/labeling")

        drop_cols_test = sorted(set(
            [c for c in ([key] + obj_cols + extra_drop) if c in test_df.columns]
            + [c for c in ID_LIKE if c in test_df.columns]
        ))

        Xte = test_df.drop(columns=drop_cols_test, errors="ignore")
        yte = test_df["y"].astype(int)

        Xte = filter_to_selected_features(Xte, selected_features)
        Xte = Xte.replace([np.inf, -np.inf], np.nan)
        Xte = align_test_to_train_cols(Xtr, Xte)
        ids_te = test_df.loc[Xte.index, key].astype(str)

        w_full = None
        if USE_SAMPLE_WEIGHTS:
            w_full = _make_sample_weights_from_strength(train_strength_all, WEIGHT_ALPHA)

        print(f"[{_ts()}] [test] External test shape: {Xte.shape}", flush=True)

        for name, mdl in models.items():
            print(f"[{_ts()}] [train] Training {name} on full train...", flush=True)
            pipe = Pipeline([("pre", pre), ("model", mdl)])
            fit_kwargs = {"model__sample_weight": w_full} if w_full is not None else {}
            pipe.fit(Xtr, ytr, **fit_kwargs)
            results[name] = _eval_and_save(name, pipe, Xte, yte, ids_te, len(ytr))
            results[name]["note"] = "external test"

    pd.DataFrame.from_dict(results, orient="index").reset_index(drop=False) \
        .rename(columns={"index": "model"}) \
        .to_csv(run_dir / f"results_clf_{run_tag}.csv", index=False)

    return results


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)
    print(f"[{_ts()}] 🎯 CLASSIFICATION  TAU={TAU_BINARY}  EPS={EPSILON_GAP}", flush=True)
    print(f"[{_ts()}] decision_rule=LP-biased (delta > -TAU_BINARY)", flush=True)
    print(f"[{_ts()}] unbounded kept for labels, excluded from delta/comparable strength", flush=True)
    print(f"[{_ts()}] sample_weights={USE_SAMPLE_WEIGHTS}  weight_alpha={WEIGHT_ALPHA}", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

    SELECTED_FEATURES = load_features_from_txt(FEATURE_FILE)

    if MODE == "binary":
        if len(RELAXATION_PAIR) != 2:
            raise ValueError(f"Binary mode requires 2 relaxations, got {len(RELAXATION_PAIR)}")
        ACTIVE_RELAXATIONS = list(RELAXATION_PAIR)
        print(f"[{_ts()}]   RELAXATION_PAIR={RELAXATION_PAIR}", flush=True)
    elif MODE == "multiclass":
        if len(RELAXATIONS) < 3:
            raise ValueError(f"Multiclass requires 3+ relaxations, got {len(RELAXATIONS)}")
        ACTIVE_RELAXATIONS = RELAXATIONS
        print(f"[{_ts()}]   RELAXATIONS={RELAXATIONS}", flush=True)
    else:
        raise ValueError(f"Unknown MODE: {MODE}")

    required_obj_cols = [f"{r}_obj" for r in ACTIVE_RELAXATIONS]
    train_tag = run_tag_from_lists(TRAIN_FEATURES)
    test_tag  = run_tag_from_lists(TEST_FEATURES) if not USE_HOLDOUT_FROM_TRAIN else "holdout"

    if MODE == "binary":
        RUN_TAG = f"{SETUP}__{MODE}__{ACTIVE_RELAXATIONS[0]}v{ACTIVE_RELAXATIONS[1]}__train[{train_tag}]__test[{test_tag}]"
    else:
        RUN_TAG = f"{SETUP}__{MODE}__{'_'.join(ACTIVE_RELAXATIONS)}__train[{train_tag}]__test[{test_tag}]"

    RUN_DIR = OUTPUT_ROOT / RUN_TAG
    RUN_DIR.mkdir(exist_ok=True, parents=True)

    print(f"[{_ts()}] Loading training data...", flush=True)
    train_df = concat_pairs(
        TRAIN_FEATURES, TRAIN_OBJECTS,
        prefer_key="id_global", required_obj_cols=required_obj_cols
    )

    test_df = None
    if not USE_HOLDOUT_FROM_TRAIN:
        print(f"[{_ts()}] Loading test data...", flush=True)
        test_df = concat_pairs(
            TEST_FEATURES, TEST_OBJECTS,
            prefer_key="id_global", required_obj_cols=required_obj_cols
        )

    print(f"[{_ts()}] Train rows: {len(train_df)} | Test rows: {len(test_df) if test_df is not None else 'holdout'}", flush=True)
    print(f"[{_ts()}] Run tag : {RUN_TAG}", flush=True)
    print(f"[{_ts()}] Output  : {RUN_DIR.resolve()}", flush=True)

    results = train_eval_classification(
        train_df, test_df, KEY_COL, RUN_DIR, RUN_TAG,
        MODE, ACTIVE_RELAXATIONS, TAU_BINARY, EPSILON_GAP, SELECTED_FEATURES,
    )

    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)
    print(f"[{_ts()}] 🎯 RESULTS:", flush=True)
    for model, m in results.items():
        metric_str = f"acc={m['accuracy']:.3f} bal_acc={m['bal_acc']:.3f}"
        if MODE == "binary":
            metric_str += f" f1_macro={m['f1_macro']:.3f} auc={m.get('roc_auc', 'N/A')}"
        else:
            metric_str += f" f1_macro={m['f1_macro']:.3f}"
        print(f"[{_ts()}]   {model}: {metric_str}", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

[20:58:12] ═══════════════════════════════════════════════════
[20:58:12] 🎯 CLASSIFICATION  TAU=0.01  EPS=0.001
[20:58:12] decision_rule=LP-biased (delta > -TAU_BINARY)
[20:58:12] unbounded kept for labels, excluded from delta/comparable strength
[20:58:12] sample_weights=True  weight_alpha=0.3
[20:58:12] ═══════════════════════════════════════════════════
[20:58:12] [features] ⚠️  File not found: selected_featuresclassdummy.txt - using ALL
[20:58:12]   RELAXATION_PAIR=('lp', 'sdp_prime')
[20:58:12] Loading training data...
[20:58:19] [merge] features_5_10_10a_DI_synth.xlsx + instancesobj_5_10_10a.csv -> rows=10000
[20:58:19] Loading test data...
[20:59:01] [merge] features_10_2a_DI_synth.xlsx + instancesobj_10_2a.csv -> rows=50000
[20:59:01] Train rows: 10000 | Test rows: 50000
[20:59:01] Run tag : DI__binary__lpvsdp_prime__train[5_10]__test[10_2a_DI_synth]
[20:59:01] Output  : C:\Users\suuser\Desktop\minlplib\clf_outputs_txt_features_synthmissings\DI__binary__lpvsdp_prime__train[5_10